# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---
**Finding 1 - Content Performance Curve**

The paper showed that the content performance was highest around 61-90 days,then generally declined after 270 days. The paper also noted that the improvement seen in the 365 + day group was mainly associated with older pages that has been refershed.

**Where does the label/group come from?**

The content groups are based on content age, defined as the number of days since the content was created. The outcome being compared is the FlyRank Health Score, which combines impressions, position, CTR, and scroll depth.

**Methodology question:**

I would ask whether the pages that were refreshed were already better-performaing pages before they were updated.If tey were,this could affect the results.

It could be more useful to compare refreshed and non-refreshed pages based on previous performance.This would help us better understand whether the improvement is related to the refresh.

<br>

**Finding 2 — Click Capture by Position Tier**

The paper reports that weighted CTR decreases as content appears further down the search results. Weighted CTR is 0.423% for the Top 3 positions and 0.050% for the Deep tier, which the paper reports as an 88% drop from Top 3 to Deep.

**Where does the label/group come from?**

The groups are based on search-result position tiers: Top 3 represents positions 1–3, Page 1 represents positions 4–10, Striking Distance represents positions 11–20, Page 3–5 represents positions 21–50, and Deep represents positions above 50. The outcome is weighted CTR, calculated as total clicks divided by total impressions within each position tier.

**Methodology question:**

I would ask how many observations and pages were included in each position group, especially because the same page can have different search positions for different queries and reporting periods.

I would also check whether the same pattern appears across different clients and time periods. This would help determine whether the relationship between position and click capture is consistent across the portfolio rather than being driven by a particular client, period, or group of observations.

The weighted CTR comparison is useful for describing the portfolio pattern, but it does not by itself prove that moving a page to a higher position will cause a specific increase in CTR.





In [ ]:
# This cell is for CODE (numbers, a query, a check).

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

---

In week 5 , I evaluated my Decision Tree using a stratified random 80/20 split.In this audit,I compare those results with a time-aware split using report_date,where earlier observatiobs are used for training and later observations for testing.

The comparison shows how the measured performance changes under a different validation approach and provides more directional,decision-support evidencr about the model's performance

### Comparison

The model performed strongly under both validation approaches. However, the measured accuracy and F1 score were slightly lower with the time-aware split.

This shows that the validation approach can affect the measured performance of the model. The time-aware results provide additional directional, decision-support evidence because the model was tested on later observations rather than randomly selected observations.

In [2]:
# ML-09 — Validation and Research Claim Audit
# Honest random vs time-aware validation

!pip install -q datasets scikit-learn pandas numpy

import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. DEFINE FEATURES
# ============================================================

features = [
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]

required_columns = features + [
    "gsc_clicks",
    "gsc_impressions",
    "report_date"
]


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("Loading dataset...")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

# Use the same sample size as the Week-5 model
sample_rows = list(ds.take(50000))

df = pd.DataFrame(sample_rows)

df = df[required_columns].copy()

print("Dataset shape:", df.shape)


# ============================================================
# 3. CLEAN DATA
# ============================================================

# Convert dates
df["report_date"] = pd.to_datetime(
    df["report_date"],
    errors="coerce"
)

# Calculate CTR
# Avoid division by zero
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

# Remove invalid values
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Keep rows with valid features, date and CTR
df = df.dropna(
    subset=features + ["report_date", "ctr"]
).copy()

# Keep positive CTR observations, matching the Week-5 setup
df = df[df["ctr"] > 0].copy()

print("Eligible rows:", len(df))

print(
    "Date range:",
    df["report_date"].min(),
    "to",
    df["report_date"].max()
)


# ============================================================
# 4. RANDOM 80/20 SPLIT — ORIGINAL VALIDATION
# ============================================================

# IMPORTANT:
# This reproduces the original Week-5 style validation.
# The target threshold is calculated from the full available
# sample because this is the original random-split baseline.

random_median = df["ctr"].median()

df_random = df.copy()

df_random["target_low_ctr"] = (
    df_random["ctr"] < random_median
).astype(int)

X_random = df_random[features]
y_random = df_random["target_low_ctr"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_random,
    y_random,
    test_size=0.20,
    random_state=42,
    stratify=y_random
)

random_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_predictions = random_model.predict(
    X_test_random
)

random_accuracy = accuracy_score(
    y_test_random,
    random_predictions
)

random_precision = precision_score(
    y_test_random,
    random_predictions,
    zero_division=0
)

random_recall = recall_score(
    y_test_random,
    random_predictions,
    zero_division=0
)

random_f1 = f1_score(
    y_test_random,
    random_predictions,
    zero_division=0
)


print("\n" + "=" * 60)
print("RANDOM 80/20 SPLIT — WEEK 5 BASELINE")
print("=" * 60)

print(f"Accuracy:  {random_accuracy:.4f}")
print(f"Precision: {random_precision:.4f}")
print(f"Recall:    {random_recall:.4f}")
print(f"F1 Score:  {random_f1:.4f}")


# ============================================================
# 5. TIME-AWARE 80/20 SPLIT
# ============================================================

# Sort chronologically
time_df = df.sort_values(
    "report_date"
).reset_index(drop=True)

split_index = int(
    len(time_df) * 0.80
)

train_time_df = time_df.iloc[
    :split_index
].copy()

test_time_df = time_df.iloc[
    split_index:
].copy()


print("\n" + "=" * 60)
print("TIME-AWARE SPLIT")
print("=" * 60)

print(
    "Training period:",
    train_time_df["report_date"].min(),
    "to",
    train_time_df["report_date"].max()
)

print(
    "Testing period:",
    test_time_df["report_date"].min(),
    "to",
    test_time_df["report_date"].max()
)

print(
    "\nTraining rows:",
    len(train_time_df)
)

print(
    "Testing rows:",
    len(test_time_df)
)


# ============================================================
# 6. CREATE TARGET USING TRAINING DATA ONLY
# ============================================================

# IMPORTANT:
# The median CTR is learned ONLY from the earlier training period.
# The later test period is not used to define the threshold.

time_train_median = train_time_df["ctr"].median()

train_time_df["target_low_ctr"] = (
    train_time_df["ctr"] < time_train_median
).astype(int)

test_time_df["target_low_ctr"] = (
    test_time_df["ctr"] < time_train_median
).astype(int)


print(
    "\nTraining CTR median:",
    time_train_median
)

print("\nTraining target distribution:")

print(
    train_time_df["target_low_ctr"]
    .value_counts()
)

print("\nTesting target distribution:")

print(
    test_time_df["target_low_ctr"]
    .value_counts()
)


# ============================================================
# 7. PREPARE TIME-AWARE MODEL DATA
# ============================================================

X_train_time = train_time_df[features]
y_train_time = train_time_df["target_low_ctr"]

X_test_time = test_time_df[features]
y_test_time = test_time_df["target_low_ctr"]


# ============================================================
# 8. TRAIN TIME-AWARE DECISION TREE
# ============================================================

time_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

time_model.fit(
    X_train_time,
    y_train_time
)

time_predictions = time_model.predict(
    X_test_time
)


# ============================================================
# 9. EVALUATE TIME-AWARE MODEL
# ============================================================

time_accuracy = accuracy_score(
    y_test_time,
    time_predictions
)

time_precision = precision_score(
    y_test_time,
    time_predictions,
    zero_division=0
)

time_recall = recall_score(
    y_test_time,
    time_predictions,
    zero_division=0
)

time_f1 = f1_score(
    y_test_time,
    time_predictions,
    zero_division=0
)


print("\n" + "=" * 60)
print("TIME-AWARE 80/20 SPLIT")
print("=" * 60)

print(f"Accuracy:  {time_accuracy:.4f}")
print(f"Precision: {time_precision:.4f}")
print(f"Recall:    {time_recall:.4f}")
print(f"F1 Score:  {time_f1:.4f}")


# ============================================================
# 10. BEFORE / AFTER COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Validation approach": [
        "Random 80/20 split",
        "Time-aware 80/20 split"
    ],
    "Accuracy": [
        random_accuracy,
        time_accuracy
    ],
    "Precision": [
        random_precision,
        time_precision
    ],
    "Recall": [
        random_recall,
        time_recall
    ],
    "F1 Score": [
        random_f1,
        time_f1
    ]
})

print("\n" + "=" * 60)
print("BEFORE / AFTER COMPARISON")
print("=" * 60)

print(
    comparison.round(4)
)


# ============================================================
# 11. DIFFERENCE IN PERFORMANCE
# ============================================================

accuracy_difference = (
    time_accuracy - random_accuracy
)

f1_difference = (
    time_f1 - random_f1
)

print("\n" + "=" * 60)
print("CHANGE FROM RANDOM TO TIME-AWARE VALIDATION")
print("=" * 60)

print(
    f"Accuracy change: {accuracy_difference:+.4f}"
)

print(
    f"F1 change:       {f1_difference:+.4f}"
)

Loading dataset...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset shape: (50000, 16)
Eligible rows: 3946
Date range: 2025-01-27 00:00:00 to 2025-02-27 00:00:00

RANDOM 80/20 SPLIT — WEEK 5 BASELINE
Accuracy:  0.9013
Precision: 0.8519
Recall:    0.9634
F1 Score:  0.9042

TIME-AWARE SPLIT
Training period: 2025-01-27 00:00:00 to 2025-02-25 00:00:00
Testing period: 2025-02-25 00:00:00 to 2025-02-27 00:00:00

Training rows: 3156
Testing rows: 790

Training CTR median: 0.05263157894736842

Training target distribution:
target_low_ctr
0    1633
1    1523
Name: count, dtype: int64

Testing target distribution:
target_low_ctr
1    460
0    330
Name: count, dtype: int64

TIME-AWARE 80/20 SPLIT
Accuracy:  0.8937
Precision: 0.8821
Recall:    0.9435
F1 Score:  0.9118

BEFORE / AFTER COMPARISON
      Validation approach  Accuracy  Precision  Recall  F1 Score
0      Random 80/20 split    0.9013     0.8519  0.9634    0.9042
1  Time-aware 80/20 split    0.8937     0.8821  0.9435    0.9118

CHANGE FROM RANDOM TO TIME-AWARE VALIDATION
Accuracy change: -0.0076
F

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

---

The features used in my model were reviewed to check whether any of them could directly reveal the target variable or include information that would not realistically available at prediction time.

The target is based on CTR,which is calculated using clicks and impressions.Therefore,features closely related to search performance need additional attention.This audit identifies potential risks rather than proving a feature causes leakage.

---
Leakage audit result

The main feature requiring attention is gsc_impressions because impressions are directly used to calculate the CTR-based target. If these impressions were measured over the same period as the target, the feature could provide information about the target that would not be available at the intended prediction time. Search-position features were also reviewed because they are strongly related to CTR, although they do not directly calculate the target. This audit identifies potential leakage risks rather than proving that leakage occurred.

These features are not automatically proof of leakage, but they could make the model's performance look stronger if the information would not be available at the intended prediction time. The results should therefore be treated as directional and decision-support evidence rather than proof of future performance.

In [4]:
leakage_audit = pd.DataFrame({
    "Feature": [
        "gsc_impressions",
        "gsc_sum_position",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "ga4_users",
        "ga4_engaged_sessions",
        "ga4_total_engagement_sec",
        "sessions_organic",
        "sessions_direct",
        "sessions_referral",
        "sessions_social",
        "sessions_paid",
        "sessions_ai"
    ],

    "Leakage risk": [
        "High",
        "Medium",
        "Medium",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low"
    ],

    "Reason": [
        "Impressions are directly used in the CTR calculation used to create the target. If measured over the same period, this can create target leakage.",
        "Search position is strongly related to CTR and may contain information about the target period, but it does not directly calculate CTR.",
        "Search position is strongly related to CTR and may contain information about the target period, but it does not directly calculate CTR.",
        "Traffic measure that does not directly calculate CTR.",
        "Traffic measure that does not directly calculate CTR.",
        "User measure that does not directly calculate CTR.",
        "Engagement measure that does not directly calculate CTR.",
        "Engagement duration that does not directly calculate CTR.",
        "Traffic source measure that does not directly calculate CTR.",
        "Traffic source measure that does not directly calculate CTR.",
        "Traffic source measure that does not directly calculate CTR.",
        "Traffic source measure that does not directly calculate CTR.",
        "Traffic source measure that does not directly calculate CTR.",
        "AI traffic source measure that does not directly calculate CTR."
    ]
})

print(leakage_audit.to_string(index=False))

                 Feature Leakage risk                                                                                                                                            Reason
         gsc_impressions         High Impressions are directly used in the CTR calculation used to create the target. If measured over the same period, this can create target leakage.
        gsc_sum_position       Medium           Search position is strongly related to CTR and may contain information about the target period, but it does not directly calculate CTR.
        gsc_avg_position       Medium           Search position is strongly related to CTR and may contain information about the target period, but it does not directly calculate CTR.
           ga4_pageviews          Low                                                                                             Traffic measure that does not directly calculate CTR.
            ga4_sessions          Low                                           

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

---

## 4. Claim Rewrite

I reviewed my Week 5 conclusions and rewrote them using more careful language. The updated claims describe what was observed and measured in this dataset rather than making predictions or proving cause and effect.

| Original claim | Rewritten claim |
|---|---|
| The Decision Tree predicts low CTR effectively. | In this analysis, the Decision Tree showed measured performance in identifying low-CTR observations. |
| Search position is the most important factor affecting CTR. | In this model, search position was observed to be an important feature. This shows a directional relationship in the available data and does not prove that position alone causes changes in CTR. |
| The model can be used to predict future low-performing content. | The model provides directional, decision-support evidence for identifying patterns related to low-CTR observations in this dataset. Further validation would be needed before using it for future predictions. |
| The model performs well. | The model showed strong measured performance on the evaluated dataset, although the results changed slightly under the time-aware split. |

These rewritten claims better reflect the evidence available from this analysis. They are based on observed and measured results and should be treated as directional, decision-support evidence rather than proof of future performance or cause-and-effect relationships.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.